# Stage 2. Model Comparison

In this notebook, I compare different approaches for predicting `position_num`.

What is included:
1. Loading the data and splitting it into train/val/test sets.
2. Feature scaling.
3. Training classical ML models and PyTorch models.
4. Comparing the results using the main metrics.

I also fixed the `DataLoader` issue so that `batch_size` works correctly in PyTorch.

In [ ]:
# Basic libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import joblib
from datetime import datetime

# Warning suppression setup
warnings.filterwarnings('ignore')

# Sklearn for classical ML models
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline

# XGBoost, LightGBM, CatBoost
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print("✓ XGBoost available")
except ImportError:
    XGB_AVAILABLE = False
    print("✗ XGBoost not installed (skipping)")

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
    print("✓ LightGBM available")
except ImportError:
    LGB_AVAILABLE = False
    print("✗ LightGBM not installed (skipping)")

try:
    from catboost import CatBoostRegressor
    CATBOOST_AVAILABLE = True
    print("✓ CatBoost available")
except ImportError:
    CATBOOST_AVAILABLE = False
    print("✗ CatBoost not installed (skipping)")

# PyTorch for neural network models
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader as TorchDataLoader, TensorDataset
    import torch.nn.functional as F
    TORCH_AVAILABLE = True

    # Checking the PyTorch version
    print(f"PyTorch version: {torch.__version__}")

    # Checking GPU availability
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"✓ PyTorch available, using device: {device}")

    # For reproducibility
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)
except ImportError as e:
    TORCH_AVAILABLE = False
    print(f"✗ PyTorch is not installed: {e}")
    print("  Neural network models will be skipped")

# Setting the random seed for reproducibility
np.random.seed(42)

# Plot style configuration
plt.style.use('ggplot')
sns.set_palette("husl")
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("=" * 60)
print("LIBRARIES LOADED SUCCESSFULLY")
print("=" * 60)

In [ ]:
class F1DatasetLoader:
    """Class for loading and preparing data"""

    def __init__(self, data_path='f1_final_dataset.csv'):
        self.data_path = data_path
        self.df = None
        self.feature_columns = None
        self.target_column = 'position_num'

    def load_data(self):
        """Loading data from a CSV file"""
        try:
            self.df = pd.read_csv(self.data_path)
            print(f"✓ Data loaded from {self.data_path}")
            print(f"  Dataset size: {self.df.shape[0]} rows, {self.df.shape[1]} columns")

            # Checking whether the target variable exists
            if self.target_column not in self.df.columns:
                print(f"  Warning: target variable '{self.target_column}' not found")
                # Trying to find an alternative
                possible_targets = ['position', 'positionOrder', 'position_num', 'positionText']
                for target in possible_targets:
                    if target in self.df.columns:
                        self.target_column = target
                        print(f"  Using '{target}' as the target variable")
                        break

            return True
        except FileNotFoundError:
            print(f"✗ File {self.data_path} not found")
            print("  Creating a sample dataset for demonstration...")
            self._create_sample_data()
            return True
        except Exception as e:
            print(f"✗ Loading error: {e}")
            return False

    def _create_sample_data(self):
        """Creating a sample dataset for demonstration"""
        np.random.seed(42)
        n_samples = 5000  # Reduced for faster execution

        # Creating features
        data = {
            'grid': np.random.randint(1, 21, n_samples),
            'quali_position': np.random.randint(1, 21, n_samples),
            'driver_age': np.random.uniform(20, 40, n_samples),
            'driver_avg_history': np.random.uniform(5, 15, n_samples),
            'constructor_avg_history': np.random.uniform(5, 15, n_samples),
            'circuit_driver_avg': np.random.uniform(5, 15, n_samples),
            'circuit_constructor_avg': np.random.uniform(5, 15, n_samples),
            'last_3_avg': np.random.uniform(5, 15, n_samples),
            'quali_finish_diff': np.random.normal(0, 3, n_samples),
            'driver_races_count': np.random.randint(1, 300, n_samples),
            'constructor_races_count': np.random.randint(1, 500, n_samples),
            'prev_podium': np.random.randint(0, 2, n_samples),
            'prev_win': np.random.randint(0, 2, n_samples),
            'year': np.random.choice([2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023], n_samples),
            'round': np.random.randint(1, 23, n_samples),
            'raceId': np.arange(n_samples),
            'driverId': np.random.randint(1, 100, n_samples),
            'constructorId': np.random.randint(1, 20, n_samples),
            'circuitId': np.random.randint(1, 30, n_samples)
        }

        # Creating the target variable (finishing position)
        # Depends on grid, quali_position, and added noise
        position = (0.6 * data['grid'] +
                    0.3 * data['quali_position'] +
                    0.1 * np.random.normal(0, 2, n_samples))
        position = np.clip(position, 1, 30).astype(int)
        data['position_num'] = position

        self.df = pd.DataFrame(data)
        print(f"  Sample dataset created: {self.df.shape}")

    def get_features(self):
        """Defining features for modeling"""
        # Main features (should match stage 1)
        base_features = ['grid', 'quali_position', 'driver_age', 'driver_avg_history',
                        'constructor_avg_history', 'circuit_driver_avg',
                        'circuit_constructor_avg', 'last_3_avg', 'quali_finish_diff',
                        'driver_races_count', 'constructor_races_count',
                        'prev_podium', 'prev_win']

        # Checking which features exist in the dataset
        self.feature_columns = [col for col in base_features if col in self.df.columns]
        missing = set(base_features) - set(self.feature_columns)

        if missing:
            print(f"  Warning: missing features {missing}")

        print(f"  Using {len(self.feature_columns)} features: {self.feature_columns}")
        return self.feature_columns

    def split_by_year(self, train_years=None, val_years=None, test_years=None):
        """Splitting data by years"""
        if 'year' not in self.df.columns:
            print("  Column 'year' not found, using random split")
            train_df, temp_df = train_test_split(self.df, test_size=0.3, random_state=42)
            val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=43)
            return train_df, val_df, test_df

        if train_years is None:
            years = sorted(self.df['year'].unique())
            # Using 70% for train, 15% for validation, and 15% for test by years
            n_years = len(years)
            train_years = years[:int(n_years * 0.7)]
            val_years = years[int(n_years * 0.7):int(n_years * 0.85)]
            test_years = years[int(n_years * 0.85):]

        train_df = self.df[self.df['year'].isin(train_years)].copy()
        val_df = self.df[self.df['year'].isin(val_years)].copy() if val_years else pd.DataFrame()
        test_df = self.df[self.df['year'].isin(test_years)].copy() if test_years else pd.DataFrame()

        print(f"\nData split:")
        print(f"  Train ({min(train_years) if train_years else 'N/A'}-{max(train_years) if train_years else 'N/A'}): {len(train_df)} records")
        print(f"  Validation ({val_years}): {len(val_df)} records")
        print(f"  Test ({test_years}): {len(test_df)} records")

        return train_df, val_df, test_df

In [ ]:
if TORCH_AVAILABLE:
    class F1RegressionMLP(nn.Module):
        """
        Multilayer perceptron for finishing position regression
        """
        def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout_rate=0.2):
            super(F1RegressionMLP, self).__init__()

            layers = []
            prev_dim = input_dim

            # Hidden layers
            for hidden_dim in hidden_dims:
                layers.extend([
                    nn.Linear(prev_dim, hidden_dim),
                    nn.BatchNorm1d(hidden_dim),
                    nn.ReLU(),
                    nn.Dropout(dropout_rate)
                ])
                prev_dim = hidden_dim

            # Output layer
            layers.append(nn.Linear(prev_dim, 1))

            self.network = nn.Sequential(*layers)

        def forward(self, x):
            return self.network(x).squeeze()

    class F1ResidualBlock(nn.Module):
        """Residual block"""
        def __init__(self, dim):
            super(F1ResidualBlock, self).__init__()
            self.fc1 = nn.Linear(dim, dim)
            self.bn1 = nn.BatchNorm1d(dim)
            self.fc2 = nn.Linear(dim, dim)
            self.bn2 = nn.BatchNorm1d(dim)

        def forward(self, x):
            residual = x
            out = F.relu(self.bn1(self.fc1(x)))
            out = self.bn2(self.fc2(out))
            out = F.relu(out + residual)
            return out

    class F1ResidualNetwork(nn.Module):
        """
        Neural network with residual connections
        """
        def __init__(self, input_dim, hidden_dim=128, num_blocks=3, dropout_rate=0.2):
            super(F1ResidualNetwork, self).__init__()

            self.input_layer = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            )

            # Residual blocks
            self.residual_blocks = nn.ModuleList([
                F1ResidualBlock(hidden_dim) for _ in range(num_blocks)
            ])

            self.output_layer = nn.Linear(hidden_dim, 1)

        def forward(self, x):
            x = self.input_layer(x)
            for block in self.residual_blocks:
                x = block(x)
            return self.output_layer(x).squeeze()

    class F1AttentionModel(nn.Module):
        """
        Model with an attention mechanism
        """
        def __init__(self, input_dim, hidden_dim=64):
            super(F1AttentionModel, self).__init__()

            self.attention = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.Tanh(),
                nn.Linear(hidden_dim, input_dim),
                nn.Softmax(dim=1)
            )

            self.fc1 = nn.Linear(input_dim, hidden_dim)
            self.bn1 = nn.BatchNorm1d(hidden_dim)
            self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
            self.bn2 = nn.BatchNorm1d(hidden_dim // 2)
            self.fc3 = nn.Linear(hidden_dim // 2, 1)

            self.dropout = nn.Dropout(0.2)

        def forward(self, x):
            # Applying attention to features
            att_weights = self.attention(x)
            attended_features = x * att_weights

            # Main layers
            x = F.relu(self.bn1(self.fc1(attended_features)))
            x = self.dropout(x)
            x = F.relu(self.bn2(self.fc2(x)))
            x = self.dropout(x)
            x = self.fc3(x)

            return x.squeeze(), att_weights  # Returning attention weights as well

In [ ]:
class ModelEvaluator:
    """Class for evaluating models"""

    @staticmethod
    def calculate_metrics(y_true, y_pred):
        """Calculating all metrics"""
        return {
            'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'R2': r2_score(y_true, y_pred),
            'MAPE': mean_absolute_percentage_error(y_true, y_pred) * 100
        }

    @staticmethod
    def evaluate_model(model, X_train, y_train, X_val, y_val, model_name="Model"):
        """Full model evaluation"""
        # Predictions
        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)

        # Metrics
        train_metrics = ModelEvaluator.calculate_metrics(y_train, y_train_pred)
        val_metrics = ModelEvaluator.calculate_metrics(y_val, y_val_pred)

        # Results
        results = {
            'Model': model_name,
            'Train MAE': train_metrics['MAE'],
            'Train RMSE': train_metrics['RMSE'],
            'Train R2': train_metrics['R2'],
            'Train MAPE': train_metrics['MAPE'],
            'Val MAE': val_metrics['MAE'],
            'Val RMSE': val_metrics['RMSE'],
            'Val R2': val_metrics['R2'],
            'Val MAPE': val_metrics['MAPE'],
            'Overfitting': val_metrics['MAE'] - train_metrics['MAE']
        }

        return results

    @staticmethod
    def plot_predictions(y_true, y_pred, title="Predictions vs Actual"):
        """Prediction visualization"""
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # 1. Scatter plot
        axes[0, 0].scatter(y_true, y_pred, alpha=0.3, s=10)
        axes[0, 0].plot([y_true.min(), y_true.max()],
                        [y_true.min(), y_true.max()],
                        'r--', lw=2, label='Ideal')
        axes[0, 0].set_xlabel('Actual')
        axes[0, 0].set_ylabel('Predicted')
        axes[0, 0].set_title(f'{title}\nMAE={mean_absolute_error(y_true, y_pred):.3f}')
        axes[0, 0].legend()

        # 2. Residuals distribution
        errors = y_pred - y_true
        axes[0, 1].hist(errors, bins=30, edgecolor='black', alpha=0.7)
        axes[0, 1].axvline(0, color='red', linestyle='--', label='Zero error')
        axes[0, 1].axvline(errors.mean(), color='blue', linestyle='--',
                           label=f'Mean error: {errors.mean():.3f}')
        axes[0, 1].set_xlabel('Error')
        axes[0, 1].set_ylabel('Frequency')
        axes[0, 1].set_title('Residuals Distribution')
        axes[0, 1].legend()

        # 3. Error vs Actual
        axes[1, 0].scatter(y_true, errors, alpha=0.3, s=10)
        axes[1, 0].axhline(0, color='red', linestyle='--')
        axes[1, 0].axhline(errors.std(), color='orange', linestyle='--', alpha=0.5, label='±1 std')
        axes[1, 0].axhline(-errors.std(), color='orange', linestyle='--', alpha=0.5)
        axes[1, 0].set_xlabel('Actual')
        axes[1, 0].set_ylabel('Error')
        axes[1, 0].set_title('Error vs Actual')
        axes[1, 0].legend()

        # 4. Q-Q plot
        from scipy import stats
        stats.probplot(errors, dist="norm", plot=axes[1, 1])
        axes[1, 1].set_title('Q-Q Plot')

        plt.tight_layout()
        plt.show()

        return errors

class EarlyStopping:
    """Early stopping for PyTorch"""
    def __init__(self, patience=10, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

In [ ]:
print("=" * 60)
print("DATA LOADING")
print("=" * 60)

# Loading data
data_loader = F1DatasetLoader('f1_final_dataset.csv')
data_loader.load_data()

# Getting features
feature_columns = data_loader.get_features()
target_column = data_loader.target_column

# Splitting by years
train_df, val_df, test_df = data_loader.split_by_year()

# Checking that all splits are non-empty
if len(val_df) == 0:
    print("Warning: validation set is empty. Creating it from train...")
    train_df, val_df = train_test_split(train_df, test_size=0.15, random_state=42)

if len(test_df) == 0:
    print("Warning: test set is empty. Creating it from the remaining data...")
    train_df, test_df = train_test_split(train_df, test_size=0.15, random_state=43)

# Creating feature matrices
X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_val = val_df[feature_columns]
y_val = val_df[target_column]

X_test = test_df[feature_columns]
y_test = test_df[target_column]

print(f"\nMatrix shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  X_val: {X_val.shape}")
print(f"  y_val: {y_val.shape}")
print(f"  X_test: {X_test.shape}")
print(f"  y_test: {y_test.shape}")

In [ ]:
print("\n" + "=" * 60)
print("FEATURE SCALING")
print("=" * 60)

# Trying different scaling methods
scalers = {
    'StandardScaler': StandardScaler(),
    'RobustScaler': RobustScaler(),
    'MinMaxScaler': MinMaxScaler()
}

best_scaler_name = 'StandardScaler'
best_scaler = StandardScaler()

# Fitting on the train data
X_train_scaled = best_scaler.fit_transform(X_train)
X_val_scaled = best_scaler.transform(X_val)
X_test_scaled = best_scaler.transform(X_test)

# Converting back to DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_columns, index=X_train.index)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=feature_columns, index=X_val.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_columns, index=X_test.index)

print(f"Using {best_scaler_name}")
print(f"Means after scaling: {X_train_scaled.mean().round(2).tolist()[:5]}...")
print(f"Standard deviations: {X_train_scaled.std().round(2).tolist()[:5]}...")

In [ ]:
print("\n" + "=" * 60)
print("TRAINING CLASSICAL ML MODELS")
print("=" * 60)

all_results = []

# 7.1 Linear models
print("\n--- Linear models ---")

# Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_results = ModelEvaluator.evaluate_model(lr_model, X_train_scaled, y_train,
                                          X_val_scaled, y_val, "Linear Regression")
all_results.append(lr_results)
print(f"Linear Regression - Val MAE: {lr_results['Val MAE']:.3f}")

# Ridge Regression
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)
ridge_results = ModelEvaluator.evaluate_model(ridge_model, X_train_scaled, y_train,
                                             X_val_scaled, y_val, "Ridge Regression")
all_results.append(ridge_results)
print(f"Ridge Regression - Val MAE: {ridge_results['Val MAE']:.3f}")

# Lasso Regression
lasso_model = Lasso(alpha=0.01, max_iter=10000)
lasso_model.fit(X_train_scaled, y_train)
lasso_results = ModelEvaluator.evaluate_model(lasso_model, X_train_scaled, y_train,
                                             X_val_scaled, y_val, "Lasso Regression")
all_results.append(lasso_results)
print(f"Lasso Regression - Val MAE: {lasso_results['Val MAE']:.3f}")

# ElasticNet
elastic_model = ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000)
elastic_model.fit(X_train_scaled, y_train)
elastic_results = ModelEvaluator.evaluate_model(elastic_model, X_train_scaled, y_train,
                                               X_val_scaled, y_val, "ElasticNet")
all_results.append(elastic_results)
print(f"ElasticNet - Val MAE: {elastic_results['Val MAE']:.3f}")

# 7.2 Decision trees and ensembles
print("\n--- Decision trees and ensembles ---")

# Decision Tree
dt_model = DecisionTreeRegressor(max_depth=10, random_state=42)
dt_model.fit(X_train_scaled, y_train)
dt_results = ModelEvaluator.evaluate_model(dt_model, X_train_scaled, y_train,
                                          X_val_scaled, y_val, "Decision Tree")
all_results.append(dt_results)
print(f"Decision Tree - Val MAE: {dt_results['Val MAE']:.3f}")

# Random Forest
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)
rf_results = ModelEvaluator.evaluate_model(rf_model, X_train_scaled, y_train,
                                          X_val_scaled, y_val, "Random Forest")
all_results.append(rf_results)
print(f"Random Forest - Val MAE: {rf_results['Val MAE']:.3f}")

# Extra Trees
et_model = ExtraTreesRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
et_model.fit(X_train_scaled, y_train)
et_results = ModelEvaluator.evaluate_model(et_model, X_train_scaled, y_train,
                                          X_val_scaled, y_val, "Extra Trees")
all_results.append(et_results)
print(f"Extra Trees - Val MAE: {et_results['Val MAE']:.3f}")

# Gradient Boosting
gb_model = GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
gb_model.fit(X_train_scaled, y_train)
gb_results = ModelEvaluator.evaluate_model(gb_model, X_train_scaled, y_train,
                                          X_val_scaled, y_val, "Gradient Boosting")
all_results.append(gb_results)
print(f"Gradient Boosting - Val MAE: {gb_results['Val MAE']:.3f}")

# AdaBoost
ada_model = AdaBoostRegressor(n_estimators=100, random_state=42)
ada_model.fit(X_train_scaled, y_train)
ada_results = ModelEvaluator.evaluate_model(ada_model, X_train_scaled, y_train,
                                           X_val_scaled, y_val, "AdaBoost")
all_results.append(ada_results)
print(f"AdaBoost - Val MAE: {ada_results['Val MAE']:.3f}")

# 7.3 KNN
print("\n--- KNN ---")

# KNN
knn_model = KNeighborsRegressor(n_neighbors=10)
knn_model.fit(X_train_scaled, y_train)
knn_results = ModelEvaluator.evaluate_model(knn_model, X_train_scaled, y_train,
                                           X_val_scaled, y_val, "KNN")
all_results.append(knn_results)
print(f"KNN - Val MAE: {knn_results['Val MAE']:.3f}")

# 7.4 XGBoost
if XGB_AVAILABLE:
    print("\n--- XGBoost ---")
    xgb_model = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
    xgb_model.fit(X_train_scaled, y_train)
    xgb_results = ModelEvaluator.evaluate_model(xgb_model, X_train_scaled, y_train,
                                                X_val_scaled, y_val, "XGBoost")
    all_results.append(xgb_results)
    print(f"XGBoost - Val MAE: {xgb_results['Val MAE']:.3f}")

# 7.5 LightGBM
if LGB_AVAILABLE:
    print("\n--- LightGBM ---")
    lgb_model = lgb.LGBMRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, verbose=-1)
    lgb_model.fit(X_train_scaled, y_train)
    lgb_results = ModelEvaluator.evaluate_model(lgb_model, X_train_scaled, y_train,
                                                X_val_scaled, y_val, "LightGBM")
    all_results.append(lgb_results)
    print(f"LightGBM - Val MAE: {lgb_results['Val MAE']:.3f}")

# 7.6 CatBoost
if CATBOOST_AVAILABLE:
    print("\n--- CatBoost ---")
    cat_model = CatBoostRegressor(n_estimators=100, depth=6, learning_rate=0.1, random_state=42, verbose=0)
    cat_model.fit(X_train_scaled, y_train)
    cat_results = ModelEvaluator.evaluate_model(cat_model, X_train_scaled, y_train,
                                                X_val_scaled, y_val, "CatBoost")
    all_results.append(cat_results)
    print(f"CatBoost - Val MAE: {cat_results['Val MAE']:.3f}")

In [ ]:
if TORCH_AVAILABLE:
    print("\n" + "=" * 60)
    print("TRAINING PYTORCH MODELS")
    print("=" * 60)

    def prepare_pytorch_data(X_train, y_train, X_val, y_val, batch_size=32):
        """Preparing data for PyTorch with correct DataLoader parameters"""
        X_train_tensor = torch.FloatTensor(X_train.values)
        y_train_tensor = torch.FloatTensor(y_train.values)
        X_val_tensor = torch.FloatTensor(X_val.values)
        y_val_tensor = torch.FloatTensor(y_val.values)

        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

        train_loader = TorchDataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = TorchDataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        return train_loader, val_loader

    def train_pytorch_model(model, train_loader, val_loader, criterion, optimizer,
                           epochs=100, device='cpu', model_name="PyTorch"):
        """Training a PyTorch model"""
        train_losses = []
        val_losses = []
        best_val_loss = float('inf')
        early_stopping = EarlyStopping(patience=15)

        for epoch in range(epochs):
            # Training
            model.train()
            train_loss = 0
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)

                optimizer.zero_grad()
                if model_name == "Attention":
                    outputs, _ = model(batch_X)
                else:
                    outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()

                train_loss += loss.item()

            avg_train_loss = train_loss / len(train_loader)
            train_losses.append(avg_train_loss)

            # Validation
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for batch_X, batch_y in val_loader:
                    batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                    if model_name == "Attention":
                        outputs, _ = model(batch_X)
                    else:
                        outputs = model(batch_X)
                    loss = criterion(outputs, batch_y)
                    val_loss += loss.item()

            avg_val_loss = val_loss / len(val_loader)
            val_losses.append(avg_val_loss)

            # Save best model
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                torch.save(model.state_dict(), f'best_{model_name}.pth')

            # Early stopping
            early_stopping(avg_val_loss)
            if early_stopping.early_stop:
                print(f"Early stopping at epoch {epoch+1}")
                break

            if (epoch + 1) % 20 == 0:
                print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')

        return train_losses, val_losses

    def evaluate_pytorch_model(model, data_loader, device='cpu', model_name="PyTorch"):
        """Evaluating a PyTorch model"""
        model.eval()
        predictions = []
        actuals = []
        attention_weights = []

        with torch.no_grad():
            for batch_X, batch_y in data_loader:
                batch_X = batch_X.to(device)
                if model_name == "Attention":
                    outputs, att_weights = model(batch_X)
                    attention_weights.extend(att_weights.cpu().numpy())
                else:
                    outputs = model(batch_X)
                predictions.extend(outputs.cpu().numpy())
                actuals.extend(batch_y.numpy())

        if attention_weights:
            return np.array(predictions), np.array(actuals), np.array(attention_weights)
        return np.array(predictions), np.array(actuals), None

    # Data preparation
    batch_size = 32
    train_loader, val_loader = prepare_pytorch_data(
        X_train_scaled, y_train, X_val_scaled, y_val, batch_size
    )
    input_dim = X_train_scaled.shape[1]

    # 8.1 MLP Model
    print("\n--- MLP Model ---")
    mlp_model = F1RegressionMLP(input_dim=input_dim, hidden_dims=[128, 64, 32], dropout_rate=0.2).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(mlp_model.parameters(), lr=0.001, weight_decay=1e-5)

    mlp_train_losses, mlp_val_losses = train_pytorch_model(
        mlp_model, train_loader, val_loader, criterion, optimizer,
        epochs=100, device=device, model_name="MLP"
    )

    mlp_model.load_state_dict(torch.load('best_MLP.pth'))
    y_val_pred_mlp, y_val_actual, _ = evaluate_pytorch_model(mlp_model, val_loader, device)

    mlp_metrics = ModelEvaluator.calculate_metrics(y_val_actual, y_val_pred_mlp)
    mlp_results = {
        'Model': 'PyTorch MLP',
        'Train MAE': np.mean(mlp_train_losses[-10:]) ** 0.5,  # Approximate
        'Val MAE': mlp_metrics['MAE'],
        'Val RMSE': mlp_metrics['RMSE'],
        'Val R2': mlp_metrics['R2'],
        'Val MAPE': mlp_metrics['MAPE'],
        'Overfitting': 0  # Placeholder
    }
    all_results.append(mlp_results)
    print(f"MLP - Val MAE: {mlp_metrics['MAE']:.3f}")

    # 8.2 ResNet Model
    print("\n--- ResNet Model ---")
    resnet_model = F1ResidualNetwork(input_dim=input_dim, hidden_dim=128, num_blocks=3, dropout_rate=0.2).to(device)
    optimizer_res = optim.Adam(resnet_model.parameters(), lr=0.0005, weight_decay=1e-5)

    resnet_train_losses, resnet_val_losses = train_pytorch_model(
        resnet_model, train_loader, val_loader, criterion, optimizer_res,
        epochs=100, device=device, model_name="ResNet"
    )

    resnet_model.load_state_dict(torch.load('best_ResNet.pth'))
    y_val_pred_resnet, y_val_actual, _ = evaluate_pytorch_model(resnet_model, val_loader, device)

    resnet_metrics = ModelEvaluator.calculate_metrics(y_val_actual, y_val_pred_resnet)
    resnet_results = {
        'Model': 'PyTorch ResNet',
        'Train MAE': np.mean(resnet_train_losses[-10:]) ** 0.5,
        'Val MAE': resnet_metrics['MAE'],
        'Val RMSE': resnet_metrics['RMSE'],
        'Val R2': resnet_metrics['R2'],
        'Val MAPE': resnet_metrics['MAPE'],
        'Overfitting': 0
    }
    all_results.append(resnet_results)
    print(f"ResNet - Val MAE: {resnet_metrics['MAE']:.3f}")

    # 8.3 Attention Model
    print("\n--- Attention Model ---")
    attention_model = F1AttentionModel(input_dim=input_dim, hidden_dim=64).to(device)
    optimizer_att = optim.AdamW(attention_model.parameters(), lr=0.001, weight_decay=0.01)

    att_train_losses, att_val_losses = train_pytorch_model(
        attention_model, train_loader, val_loader, criterion, optimizer_att,
        epochs=100, device=device, model_name="Attention"
    )

    attention_model.load_state_dict(torch.load('best_Attention.pth'))
    y_val_pred_att, y_val_actual, att_weights = evaluate_pytorch_model(
        attention_model, val_loader, device, model_name="Attention"
    )

    att_metrics = ModelEvaluator.calculate_metrics(y_val_actual, y_val_pred_att)
    att_results = {
        'Model': 'PyTorch Attention',
        'Train MAE': np.mean(att_train_losses[-10:]) ** 0.5,
        'Val MAE': att_metrics['MAE'],
        'Val RMSE': att_metrics['RMSE'],
        'Val R2': att_metrics['R2'],
        'Val MAPE': att_metrics['MAPE'],
        'Overfitting': 0
    }
    all_results.append(att_results)
    print(f"Attention - Val MAE: {att_metrics['MAE']:.3f}")

    # 8.4 Ensemble
    print("\n--- Ensemble Model ---")
    ensemble_pred = (y_val_pred_mlp + y_val_pred_resnet + y_val_pred_att) / 3
    ensemble_metrics = ModelEvaluator.calculate_metrics(y_val_actual, ensemble_pred)
    ensemble_results = {
        'Model': 'PyTorch Ensemble',
        'Train MAE': (mlp_results['Train MAE'] + resnet_results['Train MAE'] + att_results['Train MAE']) / 3,
        'Val MAE': ensemble_metrics['MAE'],
        'Val RMSE': ensemble_metrics['RMSE'],
        'Val R2': ensemble_metrics['R2'],
        'Val MAPE': ensemble_metrics['MAPE'],
        'Overfitting': 0
    }
    all_results.append(ensemble_results)
    print(f"Ensemble - Val MAE: {ensemble_metrics['MAE']:.3f}")

In [ ]:
# Summary table and visual model comparison
results_df = pd.DataFrame(all_results).copy()
results_df = results_df.sort_values('Val MAE').reset_index(drop=True)

print('Top 10 models by Validation MAE:')
print(results_df[['Model', 'Val MAE', 'Val RMSE', 'Val R2', 'Overfitting']].head(10).to_string(index=False))

plt.figure(figsize=(12, 6))
plot_df = results_df.head(12)
sns.barplot(data=plot_df, x='Val MAE', y='Model', palette='mako')
plt.title('Model comparison (lower Val MAE is better)')
plt.xlabel('Validation MAE')
plt.ylabel('Model')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.scatter(results_df['Val MAE'], results_df['Overfitting'], alpha=0.7)
for _, row in results_df.head(5).iterrows():
    plt.annotate(row['Model'], (row['Val MAE'], row['Overfitting']), fontsize=9)
plt.axhline(0, color='gray', linestyle='--', linewidth=1)
plt.title('Chart: quality vs overfitting')
plt.xlabel('Validation MAE')
plt.ylabel('Overfitting (Val MAE - Train MAE)')
plt.tight_layout()
plt.show()

## Short Conclusion

This stage provides a baseline comparison of models and a reference point for building my own neural network in stage 3.